| STT | Tên | MSSV | Đánh giá tiến độ |
|-----|-----|------|------------------|
| 1   |  Huỳnh Thế Hy   | 051205009083     |       100%           |
| 2   |   Nghiêm Đức Thuận  | 060205003756     |       100%           |
| 3   |   Đào Thiện Nhân  | 052205008343     |             100%     |
| 4   |  Tạ Nguyễn Quốc Triệu   | 051205004949    |            100%      |
| 5   | Hoàng Phú    |  2251120373    |         100%         |
| 6   | Thanh Tân    |079205011306|             100%     |


# So sánh ảnh bằng Wavelet Transform

**Mục tiêu:** Xây dựng hệ thống so sánh sự tương đồng giữa các hình ảnh sử dụng biến đổi Wavelet (DWT).

**Phương pháp:** Sử dụng Wavelet Hash (wHash) để chuyển ảnh thành chuỗi bit và so sánh bằng khoảng cách Hamming.

## 1. Import thư viện và cấu hình

In [1]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pywt

from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_curve, auc
)

warnings.filterwarnings("ignore")

# Cấu hình đường dẫn
BASE_DIR  = "data/DWT"
CSV_PATH  = "data/DWT/pairs.csv"
IMAGE_DIR = "."

if not os.path.exists(BASE_DIR) and os.path.exists(os.path.join("..", BASE_DIR)):
    BASE_DIR  = os.path.join("..", BASE_DIR)
    CSV_PATH  = os.path.join("..", CSV_PATH)
    IMAGE_DIR = ".."

# Tham số toàn cục
RANDOM_SEED   = 42
HASH_SIZE     = 8
DEFAULT_WAVELET = "haar"
SAMPLE_N      = 5

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Cấu hình biểu đồ
plt.rcParams["figure.dpi"]      = 110
plt.rcParams["axes.titlesize"]  = 13
plt.rcParams["axes.labelsize"]  = 11
plt.rcParams["font.family"]     = "DejaVu Sans"
sns.set_style("whitegrid")

print("Cấu hình hoàn tất.")

Cấu hình hoàn tất.


## 2. Tiền xử lý dữ liệu

Đọc dữ liệu từ file CSV và kiểm tra tính hợp lệ của các đường dẫn ảnh.

In [2]:
# Nhận diện các cột trong DataFrame
def detect_columns(df: pd.DataFrame):
    cols = [c.lower().strip() for c in df.columns]
    col_map = dict(zip(cols, df.columns))

    label_candidates = ["label", "similar", "same", "match", "class", "target", "y", "is_same"]
    label_col = next((col_map[c] for c in label_candidates if c in col_map), df.columns[-1])

    remaining = [c for c in df.columns if c != label_col]
    if len(remaining) >= 2:
        return remaining[0], remaining[1], label_col
    raise ValueError("CSV không đủ cột dữ liệu.")

# Chuẩn hóa nhãn về 0 hoặc 1
def normalize_label(val) -> int:
    if pd.isna(val): return None
    s = str(val).strip().lower()
    if s in ("1", "true", "same", "similar", "yes", "match"): return 1
    if s in ("0", "false", "different", "dissimilar", "no", "nomatch"): return 0
    try: return int(float(s))
    except: return None

# Đọc và kiểm tra file CSV
def load_and_validate_csv(csv_path: str, image_dir: str) -> pd.DataFrame:
    print("Đọc file CSV...")
    df_raw = pd.read_csv(csv_path)
    img1_col, img2_col, label_col = detect_columns(df_raw)
    
    df = df_raw[[img1_col, img2_col, label_col]].copy()
    df.columns = ["img1", "img2", "label_raw"]
    
    valid_rows = []
    for idx, row in df.iterrows():
        label = normalize_label(row["label_raw"])
        if label not in (0, 1): continue
        
        p1 = os.path.join(image_dir, str(row["img1"]).strip())
        p2 = os.path.join(image_dir, str(row["img2"]).strip())
        if os.path.exists(p1) and os.path.exists(p2):
            valid_rows.append({"img1": p1, "img2": p2, "label": label})

    df_clean = pd.DataFrame(valid_rows)
    print(f"Số dòng hợp lệ: {len(df_clean)}/{len(df_raw)}")
    return df_clean

df_pairs = load_and_validate_csv(CSV_PATH, IMAGE_DIR)

# Chia tập Train/Test
df_train, df_test = train_test_split(
    df_pairs, test_size=0.2, random_state=RANDOM_SEED, stratify=df_pairs['label']
)
print(f"Train: {len(df_train)}, Test: {len(df_test)}")

Đọc file CSV...


FileNotFoundError: [Errno 2] No such file or directory: 'data/DWT/pairs.csv'

In [ ]:
# Thống kê tập dữ liệu
n_similar = (df_pairs["label"] == 1).sum()
n_dissim  = (df_pairs["label"] == 0).sum()

plt.figure(figsize=(5, 3.5))
plt.bar(["Similar", "Dissimilar"], [n_similar, n_dissim], color=["steelblue", "tomato"])
plt.title("Phân phối nhãn")
plt.show()

In [ ]:
# Hiển thị các cặp ảnh mẫu
def show_sample_pairs(df: pd.DataFrame, n: int = 5, seed: int = RANDOM_SEED):
    sample = df.sample(n=min(n, len(df)), random_state=seed).reset_index(drop=True)
    fig, axes = plt.subplots(n, 2, figsize=(7, n * 2.8))

    for i, row in sample.iterrows():
        label_str = "Similar" if row["label"] == 1 else "Dissimilar"
        color = "green" if row["label"] == 1 else "red"
        for j, img_path in enumerate([row["img1"], row["img2"]]):
            ax = axes[i][j] if n > 1 else axes[j]
            img = Image.open(img_path).convert("RGB")
            ax.imshow(img)
            ax.set_title(f"Cặp {i+1} - [{label_str}]", color=color, fontsize=9)
            ax.axis("off")
    plt.tight_layout()
    plt.show()

show_sample_pairs(df_pairs, n=SAMPLE_N)

## 3. Thuật toán Wavelet Hash

Chuyển đổi ảnh sang không gian Wavelet, lấy các hệ số tần số thấp (LL) để tạo mã hash.

In [ ]:
# Hàm tạo Wavelet Hash
def wavelet_hash(image_path: str, hash_size: int = HASH_SIZE, wavelet: str = DEFAULT_WAVELET):
    try:
        img = Image.open(image_path).convert("L")
        img = img.resize((hash_size * 2, hash_size * 2), Image.Resampling.LANCZOS)
        pixels = np.asarray(img, dtype=np.float32)

        # Biến đổi DWT 2D
        coeffs = pywt.dwt2(pixels, wavelet)
        LL, _ = coeffs
        LL = LL[:hash_size, :hash_size]

        # Tạo hash bit
        return (LL > LL.mean()).flatten()
    except:
        return None

# Tính khoảng cách Hamming
def hamming_distance(hash1, hash2):
    return int(np.sum(hash1 != hash2))

# Chuyển sang độ tương đồng (0-1)
def hamming_to_similarity(dist, hash_length):
    return 1.0 - dist / hash_length

## 4. Tính toán độ tương đồng

Tính toán mã hash và độ tương đồng cho toàn bộ các cặp ảnh.

In [ ]:
def compute_all_similarities(df, hash_size=HASH_SIZE, wavelet=DEFAULT_WAVELET):
    hash_length = hash_size * hash_size
    results = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {wavelet}"):
        h1 = wavelet_hash(row["img1"], hash_size, wavelet)
        h2 = wavelet_hash(row["img2"], hash_size, wavelet)
        if h1 is not None and h2 is not None:
            dist = hamming_distance(h1, h2)
            results.append({
                "img1": row["img1"], "img2": row["img2"], "label": row["label"],
                "hamming_dist": dist, "similarity": hamming_to_similarity(dist, hash_length)
            })
    return pd.DataFrame(results)

df_result_train = compute_all_similarities(df_train)

In [ ]:
# Tìm ngưỡng (threshold) tối ưu dựa trên F1-score
thresholds = np.arange(0.0, 1.01, 0.01)
y_true = df_result_train["label"].values
y_score = df_result_train["similarity"].values

best_f1, BEST_THRESHOLD = 0, 0.5
for t in thresholds:
    f1 = f1_score(y_true, (y_score >= t).astype(int), zero_division=0)
    if f1 > best_f1:
        best_f1, BEST_THRESHOLD = f1, round(t, 2)

print(f"Ngưỡng tối ưu: {BEST_THRESHOLD} (F1: {best_f1:.4f})")

## 5. Đánh giá trên tập dữ liệu Test

In [ ]:
df_result_test = compute_all_similarities(df_test)

In [ ]:
# Tính toán các chỉ số đánh giá
y_true = df_result_test["label"].values
y_score = df_result_test["similarity"].values
y_pred = (y_score >= BEST_THRESHOLD).astype(int)

print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
print(f"F1-score:  {f1_score(y_true, y_pred):.4f}")
print(f"AUC:       {auc(*roc_curve(y_true, y_score)[:2]):.4f}")

In [ ]:
# Vẽ ma trận nhầm lẫn
plt.figure(figsize=(4.5, 4))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

## 6. Trực quan hóa kết quả

In [ ]:
# Phân phối độ tương đồng
plt.figure(figsize=(10, 4))
sns.histplot(data=df_result_test, x="similarity", hue="label", kde=True, bins=30)
plt.axvline(BEST_THRESHOLD, color="black", linestyle="--")
plt.title("Phân phối Similarity Score")
plt.show()

In [ ]:
# Đường cong ROC
fpr, tpr, _ = roc_curve(y_true, y_score)
plt.plot(fpr, tpr, label=f"AUC = {auc(fpr, tpr):.3f}")
plt.plot([0, 1], [0, 1], "--")
plt.legend()
plt.title("ROC Curve")
plt.show()

## 7. So sánh các loại Wavelet

Thử nghiệm hiệu suất với nhiều loại Wavelet khác nhau.

In [ ]:
def evaluate_wavelet(df_train, df_test, wavelet):
    res_train = compute_all_similarities(df_train, wavelet=wavelet)
    res_test  = compute_all_similarities(df_test, wavelet=wavelet)
    
    # Tìm ngưỡng tốt nhất trên train
    best_t, best_f1 = 0.5, 0
    for t in np.arange(0.0, 1.01, 0.01):
        f1 = f1_score(res_train["label"], (res_train["similarity"] >= t).astype(int), zero_division=0)
        if f1 > best_f1: best_f1, best_t = f1, t
    
    # Đánh giá trên test
    y_t, y_s = res_test["label"], res_test["similarity"]
    return {
        "wavelet": wavelet, "f1": f1_score(y_t, (y_s >= best_t).astype(int), zero_division=0),
        "auc": auc(*roc_curve(y_t, y_s)[:2])
    }

results = [evaluate_wavelet(df_train, df_test, w) for w in ["haar", "db1", "db2", "db4", "sym2"]]
df_compare = pd.DataFrame(results).sort_values("f1", ascending=False)

In [ ]:
print("Bảng so sánh Wavelet:")
print(df_compare.to_string(index=False))

## 8. Kết luận

Wavelet Hash là một phương pháp nhanh và hiệu quả để so sánh độ tương đồng ảnh mà không cần model phức tạp.